# 📡 Sensor Data Simulator
## Simulated Accelerometer & Gyroscope for Fall Detection System

**Purpose**: This notebook simulates IoT sensor data (accelerometer and gyroscope) for the fall detection system.

**Role**: Data generation only - no control logic

**Note**: Use the Operator Interface to control training/inference modes

## ⚙️ Configuration

In [ ]:
# ========== MQTT BROKER CONFIGURATION ==========
JETSON_IP = "192.168.15.116"  # ⚠️ CHANGE THIS to your Jetson IP
MQTT_PORT = 1883

# ========== DEVICE CONFIGURATION ==========
DEVICE_CONFIG = {
    "house_id": "house_001",
    "sensor_id": "sensor_accel_bedroom_001",
    "device_name": "Bedroom Sensor Alice",
    "location": "master_bedroom",
    "resident_name": "Alice",
    "sensor_type": "accelerometer",
    "firmware_version": "1.2.3"
}

# ========== SENSOR PARAMETERS ==========
SAMPLING_RATE_HZ = 10  # 10 Hz = 10 readings per second
INTERVAL = 1.0 / SAMPLING_RATE_HZ  # Time between readings

print("✅ Configuration loaded")
print(f"   Jetson IP: {JETSON_IP}")
print(f"   Device: {DEVICE_CONFIG['device_name']}")
print(f"   Sensor ID: {DEVICE_CONFIG['sensor_id']}")
print(f"   Sampling Rate: {SAMPLING_RATE_HZ} Hz")

## 📦 Import Libraries

In [ ]:
import paho.mqtt.client as mqtt
import numpy as np
import json
import time
from datetime import datetime
import random
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully")

## 🔧 Sensor Simulator Class

In [ ]:
class SensorSimulator:
    """
    Simulates accelerometer and gyroscope sensor data.
    Generates realistic sensor readings for normal activity and falls.
    """
    
    def __init__(self, mqtt_client, device_config):
        self.client = mqtt_client
        self.device_config = device_config
        self.messages_sent = 0
        self.power_level = device_config.get("initial_power", 100.0)
    
    def generate_normal_reading(self):
        """Generate normal activity sensor reading"""
        # Normal accelerometer values (slight movement)
        accel_x = np.random.normal(0, 0.2)
        accel_y = np.random.normal(0, 0.2)
        accel_z = np.random.normal(9.81, 0.3)  # Gravity
        
        # Normal gyroscope values (minimal rotation)
        gyro_roll = np.random.normal(0, 5)
        gyro_pitch = np.random.normal(0, 5)
        gyro_yaw = np.random.normal(0, 5)
        
        return accel_x, accel_y, accel_z, gyro_roll, gyro_pitch, gyro_yaw
    
    def generate_fall_reading(self):
        """Generate fall event sensor reading"""
        # High acceleration during fall
        accel_x = np.random.uniform(-15, 15)
        accel_y = np.random.uniform(-15, 15)
        accel_z = np.random.uniform(-20, 5)  # Significant drop
        
        # High rotation during fall
        gyro_roll = np.random.uniform(-90, 90)
        gyro_pitch = np.random.uniform(-90, 90)
        gyro_yaw = np.random.uniform(-90, 90)
        
        return accel_x, accel_y, accel_z, gyro_roll, gyro_pitch, gyro_yaw
    
    def create_payload(self, is_fall=False, operational_mode="inference", ground_truth=None):
        """
        Create MQTT payload with sensor data.
        
        Args:
            is_fall: Whether to simulate a fall
            operational_mode: "training" or "inference"
            ground_truth: Optional ground truth labels for training mode
        """
        # Generate sensor readings
        if is_fall:
            accel_x, accel_y, accel_z, roll, pitch, yaw = self.generate_fall_reading()
        else:
            accel_x, accel_y, accel_z, roll, pitch, yaw = self.generate_normal_reading()
        
        # Simulate power drain
        self.power_level = max(0, self.power_level - 0.01)
        
        # Build payload
        payload = {
            "metadata": {
                "house_id": self.device_config["house_id"],
                "sensor_id": self.device_config["sensor_id"],
                "application": "fall_detection",
                "timestamp": datetime.now().isoformat(),
                "operational_mode": operational_mode,
                "data_purpose": "training" if operational_mode == "training" else "production"
            },
            "device_info": {
                "sensor_type": self.device_config["sensor_type"],
                "location": self.device_config["location"],
                "power_level": round(self.power_level, 2),
                "firmware_version": self.device_config["firmware_version"],
                "sensor_status": "active"
            },
            "sensor_data": {
                "accel_x": round(accel_x, 4),
                "accel_y": round(accel_y, 4),
                "accel_z": round(accel_z, 4),
                "gyro_roll": round(roll, 4),
                "gyro_pitch": round(pitch, 4),
                "gyro_yaw": round(yaw, 4)
            }
        }
        
        # Add ground truth for training mode
        if operational_mode == "training" and ground_truth is not None:
            payload["ground_truth"] = ground_truth
        
        return payload
    
    def get_topic(self, operational_mode="inference"):
        """Get appropriate MQTT topic based on operational mode"""
        house_id = self.device_config["house_id"]
        sensor_id = self.device_config["sensor_id"]
        
        if operational_mode == "training":
            return f"iot/training/{house_id}/fall_detection/accelerometer/{sensor_id}"
        else:
            return f"iot/{house_id}/fall_detection/accelerometer/{sensor_id}"
    
    def send_reading(self, is_fall=False, operational_mode="inference", ground_truth=None):
        """Send a single sensor reading"""
        payload = self.create_payload(is_fall, operational_mode, ground_truth)
        topic = self.get_topic(operational_mode)
        
        self.client.publish(topic, json.dumps(payload), qos=1)
        self.messages_sent += 1
        
        return is_fall

print("✅ SensorSimulator class defined")

## 🔌 Connect to MQTT Broker

In [ ]:
# Create MQTT client
mqtt_client = mqtt.Client(client_id="sensor_simulator")

# Connect to broker
try:
    mqtt_client.connect(JETSON_IP, MQTT_PORT, 60)
    mqtt_client.loop_start()
    time.sleep(1)
    print(f"✅ Connected to MQTT Broker at {JETSON_IP}:{MQTT_PORT}")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Please check Jetson IP and ensure MQTT broker is running")

# Create sensor simulator
DEVICE_CONFIG["initial_power"] = 100.0
simulator = SensorSimulator(mqtt_client, DEVICE_CONFIG)
print("✅ Sensor simulator ready")

---
# 🎯 Simulation Modes

## Mode 1: Training Data Generation
Use this to generate labeled data for model training

In [ ]:
def generate_training_data(num_samples=100, fall_percentage=5.0, labeled_by="operator"):
    """
    Generate labeled training data.
    
    Args:
        num_samples: Total number of samples to generate
        fall_percentage: Percentage of samples that are falls (0-100)
        labeled_by: Who labeled the data
    """
    print("\n" + "="*60)
    print("🎓 GENERATING TRAINING DATA")
    print("="*60)
    print(f"Total Samples: {num_samples}")
    print(f"Fall Percentage: {fall_percentage}%")
    print(f"Device: {DEVICE_CONFIG['device_name']}")
    print("="*60 + "\n")
    
    falls_generated = 0
    
    for i in range(num_samples):
        # Determine if this sample is a fall
        is_fall = random.random() < (fall_percentage / 100.0)
        
        # Create ground truth label
        ground_truth = {
            "is_fall": is_fall,
            "fall_detected": 1 if is_fall else 0,
            "confidence": 1.0,
            "labeled_by": labeled_by,
            "label_timestamp": datetime.now().isoformat()
        }
        
        # Send reading
        simulator.send_reading(
            is_fall=is_fall,
            operational_mode="training",
            ground_truth=ground_truth
        )
        
        if is_fall:
            falls_generated += 1
        
        # Display progress
        status = "🚨 FALL" if is_fall else "✅ Normal"
        print(f"\r{status} | Progress: {i+1}/{num_samples} | Falls: {falls_generated}",
              end='', flush=True)
        
        time.sleep(INTERVAL)
    
    print("\n\n" + "="*60)
    print("📊 TRAINING DATA GENERATION COMPLETE")
    print("="*60)
    print(f"Total Samples: {num_samples}")
    print(f"Falls: {falls_generated} ({falls_generated/num_samples*100:.1f}%)")
    print(f"Normal: {num_samples - falls_generated} ({(num_samples-falls_generated)/num_samples*100:.1f}%)")
    print("="*60)
    print("\n⚠️  Wait 30 seconds for data ingestion before starting training!")

### Run Training Data Generation

In [ ]:
# Generate 500 labeled training samples with 5% falls
generate_training_data(
    num_samples=500,
    fall_percentage=5.0,
    labeled_by="manual_operator"
)

---
## Mode 2: Inference Data Generation
Use this to generate unlabeled production data for real-time inference

In [ ]:
def generate_inference_data(duration=60, fall_probability=0.05):
    """
    Generate production/inference data.
    
    Args:
        duration: How long to run (seconds)
        fall_probability: Probability of fall per sample (0.0-1.0)
    """
    print("\n" + "="*60)
    print("📡 GENERATING INFERENCE DATA")
    print("="*60)
    print(f"Duration: {duration} seconds")
    print(f"Sampling Rate: {SAMPLING_RATE_HZ} Hz")
    print(f"Fall Probability: {fall_probability*100:.1f}%")
    print(f"Device: {DEVICE_CONFIG['device_name']}")
    print("="*60 + "\n")
    
    start_time = time.time()
    falls_simulated = 0
    samples_sent = 0
    
    try:
        while (time.time() - start_time) < duration:
            # Random fall simulation
            is_fall = random.random() < fall_probability
            
            # Send reading (no ground truth in inference mode)
            simulator.send_reading(
                is_fall=is_fall,
                operational_mode="inference"
            )
            
            if is_fall:
                falls_simulated += 1
            
            samples_sent += 1
            
            # Display
            status = "🚨 FALL" if is_fall else "✅ Normal"
            print(f"\r{status} | Samples: {samples_sent} | Falls: {falls_simulated} | "
                  f"Power: {simulator.power_level:.1f}%",
                  end='', flush=True)
            
            time.sleep(INTERVAL)
    
    except KeyboardInterrupt:
        print("\n\n⏹️  Simulation stopped by user")
    
    elapsed = time.time() - start_time
    print("\n\n" + "="*60)
    print("📊 INFERENCE DATA GENERATION COMPLETE")
    print("="*60)
    print(f"Duration: {elapsed:.1f} seconds")
    print(f"Samples Sent: {samples_sent}")
    print(f"Falls Simulated: {falls_simulated}")
    print(f"Average Rate: {samples_sent/elapsed:.1f} samples/sec")
    print("="*60)

### Run Inference Data Generation

In [ ]:
# Generate inference data for 60 seconds
generate_inference_data(
    duration=60,
    fall_probability=0.05  # 5% chance of fall per sample
)

---
## Mode 3: Manual Single Reading
Send individual readings for testing

In [ ]:
# Send a single normal reading for inference
simulator.send_reading(is_fall=False, operational_mode="inference")
print("✅ Sent normal inference reading")

In [ ]:
# Send a single fall reading for inference
simulator.send_reading(is_fall=True, operational_mode="inference")
print("🚨 Sent fall inference reading")

In [ ]:
# Send a labeled training reading
ground_truth = {
    "is_fall": True,
    "fall_detected": 1,
    "confidence": 1.0,
    "labeled_by": "manual_operator",
    "label_timestamp": datetime.now().isoformat()
}
simulator.send_reading(is_fall=True, operational_mode="training", ground_truth=ground_truth)
print("🎓 Sent labeled training reading")

---
# 🧹 Cleanup

In [ ]:
# Disconnect from MQTT broker
mqtt_client.loop_stop()
mqtt_client.disconnect()
print(f"\n✅ Disconnected from MQTT Broker")
print(f"   Total messages sent: {simulator.messages_sent}")

---
# 📝 Quick Reference

## Training Data Generation
```python
generate_training_data(num_samples=500, fall_percentage=5.0, labeled_by="operator")
```

## Inference Data Generation
```python
generate_inference_data(duration=60, fall_probability=0.05)
```

## Single Readings
```python
# Normal reading
simulator.send_reading(is_fall=False, operational_mode="inference")

# Fall reading
simulator.send_reading(is_fall=True, operational_mode="inference")

# Labeled training reading
ground_truth = {"is_fall": True, "fall_detected": 1, "confidence": 1.0}
simulator.send_reading(is_fall=True, operational_mode="training", ground_truth=ground_truth)
```

## MQTT Topics
- **Training data**: `iot/training/{house_id}/fall_detection/accelerometer/{sensor_id}`
- **Inference data**: `iot/{house_id}/fall_detection/accelerometer/{sensor_id}`

## Control the System
**⚠️ Important**: Use the **Operator Interface** (web dashboard) to:
- Start/stop training
- Switch between training and inference modes
- Monitor system status
- Manage models

This notebook only generates sensor data!